# Resume Parser API - Google Colab

Production-grade resume parsing with >90% accuracy.

This notebook demonstrates:
1. Installing dependencies
2. Running the FastAPI server in Colab
3. Uploading and parsing resumes
4. Viewing structured results

## Step 1: Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn pydantic pdfplumber spacy sentence-transformers faiss-cpu python-multipart nest-asyncio
!python -m spacy download en_core_web_trf

## Step 2: Upload Resume Parser Files

Upload all `.py` files from the `resume_parser/` directory.

In [ ]:
from google.colab import files
import os

print("Please upload the following files:")
print("  - config.py")
print("  - schemas.py")
print("  - pdf_processor.py")
print("  - entity_extractor.py")
print("  - experience_extractor.py")
print("  - skills_extractor.py")
print("  - main.py")
print("\nUploading files...")

uploaded = files.upload()
print(f"\n✅ Uploaded {len(uploaded)} files")

## Step 3: Start FastAPI Server

In [ ]:
import nest_asyncio
import uvicorn
from threading import Thread
import time

# Allow nested event loops (required for Colab)
nest_asyncio.apply()

def run_server():
    """Run FastAPI server in background thread"""
    uvicorn.run("main:app", host="0.0.0.0", port=8001, log_level="info")

# Start server in background
print("🚀 Starting Resume Parser API...")
thread = Thread(target=run_server, daemon=True)
thread.start()

# Wait for server to start
time.sleep(10)
print("✅ Server started at http://127.0.0.1:8001")
print("📚 API docs available at http://127.0.0.1:8001/docs")

## Step 4: Upload a Resume PDF

In [ ]:
from google.colab import files

print("📄 Upload a resume PDF file:")
uploaded_resume = files.upload()
resume_filename = list(uploaded_resume.keys())[0]
print(f"\n✅ Uploaded: {resume_filename}")

## Step 5: Parse Resume

In [ ]:
import requests
import json

print(f"🔍 Parsing resume: {resume_filename}...")

# Send request to API
url = "http://127.0.0.1:8001/parse"
with open(resume_filename, 'rb') as f:
    response = requests.post(url, files={"file": f})

result = response.json()

if result['success']:
    print("\n✅ Successfully parsed resume!\n")
    print("=" * 60)
    print("EXTRACTED DATA")
    print("=" * 60)
    
    data = result['data']
    
    print(f"\n👤 Name:  {data.get('name', 'N/A')}")
    print(f"📧 Email: {data.get('email', 'N/A')}")
    print(f"📱 Phone: {data.get('phone', 'N/A')}")
    
    print(f"\n💡 Skills ({len(data.get('skills', []))}):") 
    skills = data.get('skills', [])
    if skills:
        for skill in skills[:15]:  # Show first 15
            print(f"  - {skill}")
        if len(skills) > 15:
            print(f"  ... and {len(skills) - 15} more")
    
    print(f"\n💼 Experience ({len(data.get('experience', []))}):") 
    for i, exp in enumerate(data.get('experience', []), 1):
        print(f"\n  {i}. {exp.get('position', 'N/A')}")
        print(f"     Company:  {exp.get('company', 'N/A')}")
        print(f"     Duration: {exp.get('duration', 'N/A')}")
        if exp.get('description'):
            desc = exp['description'][:100]
            print(f"     Description: {desc}...")
    
    print(f"\n⏱️  Processing Time: {result.get('processing_time_ms', 0):.2f}ms")
    
else:
    print(f"\n❌ Error: {result.get('error', 'Unknown error')}")

## Step 6: View Full JSON Output

In [ ]:
print("Full JSON Response:")
print(json.dumps(result, indent=2))

## Step 7: Download Results (Optional)

In [ ]:
# Save results to JSON file
output_filename = f"{resume_filename.replace('.pdf', '')}_parsed.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2)

print(f"💾 Results saved to: {output_filename}")

# Download the file
files.download(output_filename)
print("✅ Download started")

## Additional: Batch Processing Multiple Resumes

In [ ]:
print("📄 Upload multiple resume PDFs (hold Ctrl/Cmd to select multiple):")
uploaded_resumes = files.upload()

print(f"\n✅ Uploaded {len(uploaded_resumes)} resumes")
print("\n🔍 Processing batch...\n")

batch_results = []

for filename in uploaded_resumes.keys():
    print(f"  Processing: {filename}...")
    
    with open(filename, 'rb') as f:
        response = requests.post("http://127.0.0.1:8001/parse", files={"file": f})
    
    result = response.json()
    batch_results.append({
        "filename": filename,
        "result": result
    })
    
    if result['success']:
        print(f"    ✅ Success - Found {len(result['data'].get('skills', []))} skills")
    else:
        print(f"    ❌ Failed: {result.get('error')}")

print(f"\n✅ Batch processing complete: {len(batch_results)} resumes")

# Save batch results
with open('batch_results.json', 'w', encoding='utf-8') as f:
    json.dump(batch_results, f, indent=2)

files.download('batch_results.json')
print("💾 Batch results saved and downloaded")

---

## Notes

- **First run**: Model downloads may take 2-3 minutes
- **Performance**: Colab provides sufficient resources for testing
- **Accuracy**: Expected >90% field-level accuracy on well-formatted resumes
- **Troubleshooting**: If server fails to start, restart runtime and try again

**Built with ❤️ by the Praktiki Team**